In [ ]:
"""
Long-range Ising RG with three-state couplings:
    J_r in { 0, +|J0|/r^a, -|J0|/r^a }

Parameters
----------
p      : fraction of ANTIFERROMAGNETIC (negative) couplings among NON-ZERO bonds
r_frac : fraction of ZERO couplings among all distances r = 1..D

So the probabilities per distance are
    P(J_r = 0)              = r_frac
    P(J_r = -|J0|/r^a)      = (1 - r_frac) * p
    P(J_r = +|J0|/r^a)      = (1 - r_frac) * (1 - p)

Exact fixed-composition sampling (not i.i.d. Bernoulli) is used, to match
the style of the existing staggered population-dynamics code.

Geometry: staggered two-cell geometry (imports from decimation_staggered).
RG mode: head-only, shrinking vector. No tail reconstruction.
"""

import numpy as np
import matplotlib.pyplot as plt
from numba import njit

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'lib')))

from utils import _xorshift64star_next
from decimation_staggered import (
    required_initial_max_distance,
    r_max,
    log_Rpp_Rpm,
)


# ----------------------------
# Deterministic Fisher-Yates on a length-D array (indices 1..D), Numba-safe
# ----------------------------

@njit(cache=True)
def _fisher_yates_1toD(arr, seed_mix):
    """
    In-place Fisher-Yates shuffle of arr[1..len(arr)-1], leaving arr[0] alone.
    `seed_mix` is a uint64 already mixed by the caller.
    """
    state = seed_mix
    if state == np.uint64(0):
        state = np.uint64(0xD1B54A32D192ED03)
    n = arr.shape[0] - 1  # number of shuffle-eligible entries, indices 1..n
    for i in range(n, 1, -1):
        state, rnd = _xorshift64star_next(state)
        j = 1 + int(rnd % np.uint64(i))  # j in [1, i]
        tmp = arr[i]
        arr[i] = arr[j]
        arr[j] = tmp


# ----------------------------
# Three-state coupling sign/zero pattern
# ----------------------------

@njit(cache=True)
def generate_states(D, p, r_frac, seed):
    """
    Return an int8 array `state[0..D]` with state[0]=0 unused, and for r=1..D:
        state[r] =  0 with exact count floor(r_frac * D)
        state[r] = -1 with exact count floor(p * (D - n_zero))
        state[r] = +1 with the remainder

    Then shuffled (Fisher-Yates, deterministic by seed).
    """
    state_arr = np.zeros(D + 1, dtype=np.int8)

    n_zero  = int(r_frac * D)
    n_nz    = D - n_zero
    n_minus = int(p * n_nz)
    n_plus  = n_nz - n_minus

    idx = 1
    for _ in range(n_zero):
        state_arr[idx] = 0
        idx += 1
    for _ in range(n_minus):
        state_arr[idx] = -1
        idx += 1
    for _ in range(n_plus):
        state_arr[idx] = 1
        idx += 1

    seed_mix = np.uint64(seed) ^ np.uint64(0x9E3779B97F4A7C15)
    _fisher_yates_1toD(state_arr, seed_mix)
    return state_arr


@njit(cache=True)
def build_J_dilute_signed(J0, a, D, p, r_frac, seed):
    """
    Initial coupling vector with three-state distribution.
    J[0] = 0. For r>=1, J[r] = state[r] * J0 / r^a, where state[r] in {-1,0,+1}.
    """
    states = generate_states(D, p, r_frac, seed)
    J = np.zeros(D + 1, dtype=np.float64)
    for r in range(1, D + 1):
        if states[r] != 0:
            J[r] = (float(states[r]) * J0) / (r ** a)
        # else leave as 0.0
    return J


# ----------------------------
# Diagnostics on a J vector
# ----------------------------

@njit(cache=True)
def fractions(J, zero_tol=1e-14):
    """
    Return (p_eff, r_eff) for J[1..D]:
        r_eff = fraction of |J_r| <= zero_tol
        p_eff = fraction of J_r < 0 among the non-zero entries
    """
    D = J.shape[0] - 1
    if D <= 0:
        return 0.0, 0.0
    n_zero = 0
    n_neg = 0
    n_nz = 0
    for r in range(1, D + 1):
        if abs(J[r]) <= zero_tol:
            n_zero += 1
        else:
            n_nz += 1
            if J[r] < 0.0:
                n_neg += 1
    r_eff = n_zero / D
    p_eff = (n_neg / n_nz) if n_nz > 0 else 0.0
    return p_eff, r_eff


# ----------------------------
# One RG step: staggered, head-only, shrinking vector
# ----------------------------

@njit(cache=True)
def rg_step(J):
    """
    One head-only staggered RG step. No tail reconstruction.
    Output length = r_max(D) + 1, so the vector shrinks each iteration.

    Uses exactly the same per-distance recursion as the existing code:
        J'_r = 0.5 * (log R_++ - log R_+-)
    with R_++ and R_+- coming from decimation_staggered.log_Rpp_Rpm.
    Zero entries in J are handled implicitly: a zero bond contributes e^0=1
    inside the cell-pair partition sums, so nothing special is needed here.
    """
    D = J.shape[0] - 1
    rstop = r_max(D)

    J_new = np.zeros(rstop + 1, dtype=np.float64)
    for rr in range(1, rstop + 1):
        log_pp, log_pm = log_Rpp_Rpm(rr, J)
        J_new[rr] = 0.5 * (log_pp - log_pm)
    return J_new


# ----------------------------
# RG flow
# ----------------------------

def generate_rg_flow(J0, a, D, n_steps, p, r_frac, seed=12345):
    """
    Build dilute signed initial condition and iterate `n_steps` shrinking RG steps.
    Returns a list of J vectors [J^(0), J^(1), ..., J^(n_steps)] of decreasing length.
    """
    J = build_J_dilute_signed(J0, a, D, p, r_frac, seed)
    flow = [J.copy()]
    for _ in range(n_steps):
        J = rg_step(J)
        flow.append(J.copy())
    return flow


# ----------------------------
# Plot helper
# ----------------------------

def plot_rg_flow(flow, distances_to_plot=None, title=None, ax=None):
    """
    Plot J_r vs RG step k for a list of distances. Skips a (step, r) pair
    if r exceeds the current vector length at that step (due to shrinkage).
    """
    if distances_to_plot is None:
        distances_to_plot = [1, 2, 3, 4, 5]

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    else:
        fig = ax.figure

    for rr in distances_to_plot:
        steps_ok = []
        vals = []
        for k, J in enumerate(flow):
            if 1 <= rr <= (len(J) - 1):
                steps_ok.append(k)
                vals.append(J[rr])
        if len(vals) > 0:
            ax.plot(steps_ok, vals, marker='o', label=f"$r={rr}$")

    ax.axhline(0.0, color='k', lw=0.5, alpha=0.5)
    ax.set_xlabel(r"RG iteration $k$")
    ax.set_ylabel(r"Coupling $J_r$")
    if title is not None:
        ax.set_title(title)
    ax.legend(title="distance", fontsize=9)
    fig.tight_layout()
    return fig, ax


# ----------------------------
# Simple phase-sink classifier
# ----------------------------

def classify_sink(
    flow,
    track_r=(2, 3, 4, 5),
    eval_step=None,
    thr_dis=1e-2,
    thr_ord=1e2,
):
    """
    Classify the phase sink from a flow:
      - 'disorder'  : all |J_r| < thr_dis for r in track_r
      - 'ferro'     : all  J_r > thr_ord  for r in track_r
      - 'antiferro' : all  J_r < -thr_ord for r in track_r
      - 'spinglass' : at least one |J_r| > thr_ord but signs are mixed
                      (i.e. the coupling grows but not coherently)
      - 'undetermined' : none of the above / tracked r unavailable
    """
    if eval_step is None:
        eval_step = len(flow) - 1
    eval_step = max(0, min(eval_step, len(flow) - 1))
    J_eval = flow[eval_step]
    D_eval = len(J_eval) - 1

    vals = []
    for rr in track_r:
        if not (1 <= rr <= D_eval):
            return "undetermined", {
                "reason": "missing_tracked_distance",
                "missing_r": rr, "D_eval": D_eval, "eval_step": eval_step,
            }
        vals.append(float(J_eval[rr]))
    vals = np.array(vals)
    absv = np.abs(vals)

    if np.all(absv < thr_dis):
        return "disorder", {"eval_step": eval_step, "vals": vals}
    if np.all(vals > thr_ord):
        return "ferro", {"eval_step": eval_step, "vals": vals}
    if np.all(vals < -thr_ord):
        return "antiferro", {"eval_step": eval_step, "vals": vals}
    if np.any(absv > thr_ord):
        # at least one coupling is large, but not all coherent in sign
        return "spinglass", {"eval_step": eval_step, "vals": vals}
    return "undetermined", {"eval_step": eval_step, "vals": vals}